In [0]:
# Gold layer: fact_order_items (Kimball transaction-grain fact - order lines)
# Source: b2b_ecommerce.silver.order_items at line grain, denormalized with
#         its order header b2b_ecommerce.silver.orders (LEFT join - a
#         headerless line keeps its row with null header attributes;
#         facts retain rows, they never drop them), plus surrogate-key
#         resolution against the gold dimensions.
# Grain:  exactly one row per order_item_id (the line grain) - the MERGE
#         key, so retries and re-runs are idempotent. Header fields
#         (dates, status, channel, customer) repeat on each line of the
#         same order: standard fact denormalization.
# Joins:  company_sk / product_sk resolve AS-OF the order header's
#         created_at against the SCD2 dims (dim_companies, dim_products):
#         the version open at that moment - KPI 4 needs list_price as of
#         the sale and KPI 3 category as of the sale. Fallback: an order
#         that predates the dim's first version resolves to that first
#         version (all current orders are in this bucket - the SCD2 dims
#         were first built 2026-11-18, after every order was created; as
#         future versions close, genuine as-of resolution takes over
#         automatically). customer_sk / supplier_sk are Type 1 - direct
#         joins on customer_id / supplier_id, one current row each.
#         date_sk = dim_date's smart YYYYMMDD key from order_date, NULL
#         for the 1,074 lines whose order_date_valid is false (474 null
#         + 600 future dates at line grain) - those rows are RETAINED,
#         and the net-revenue KPI views filter them via order_date_valid.
# Columns: quantity, unit_price, line_total (the measures); order_id,
#         the 5 dimension keys (date_sk, customer_sk, company_sk,
#         product_sk, supplier_sk); raw order_date + order_date_valid
#         flag (net-revenue filter, phantom-month guard); status kept
#         for ALL rows - cancelled stays in the fact (gross), net views
#         exclude it; channel (KPI 8/11 web-vs-rep conversion); header
#         created_at kept as order_created_at (SCD2 as-of key + KPI 8
#         conversion-window matching). Excluded per the 2026-11-18 KPI
#         audit (no KPI uses them): discount_pct, tax_rate, warehouse,
#         shipping_method, shipping_cost, payment_method, line_number,
#         po_number, expected_delivery_date, silver updated_at (mixed
#         ART/UTC clocks - gold excludes it by design).
# Storage: Delta-backed STREAMING TABLE (facts are stored tables, not
#         views), loaded via AUTO CDC FROM SNAPSHOT - the same
#         ingestion-time pattern as the gold dims.
# MERGE / idempotency: keyed on order_item_id (requirements' MERGE
#         constraint - safe to retry without duplicates). The snapshot
#         view is recomputed each run; an unchanged source diffs to zero
#         events and changes nothing.
# Quality: no expectations - AUTO CDC FROM SNAPSHOT flows cannot carry
#         them (same decision as the dims), and silver already owns
#         row-level validation. Pre-flight (2026-11-19) proved the exact
#         snapshot resolves exactly one SK per dim for all 30,240 lines,
#         zero header orphans, zero duplicate merge keys, and 100% of
#         the 29,166 valid order dates match a dim_date row.
# Layout: partitioned by date_sk (requirements: fact_* partitioned by
#         date; NULL partition values are valid on Delta). Delta does not
#         allow clustering keys alongside partition columns
#         (SPECIFY_CLUSTER_BY_WITH_PARTITIONED_BY_IS_NOT_ALLOWED), so no
#         cluster_by - the date partitions already prune MERGE scans and
#         the table is line-grain small (~30K rows).
from pyspark import pipelines as dp
from pyspark.sql import Window
from pyspark.sql import functions as F

GOLD = "b2b_ecommerce.gold"
SILVER_ORDERS = "b2b_ecommerce.silver.orders"
SILVER_ORDER_ITEMS = "b2b_ecommerce.silver.order_items"


# The complete fact snapshot, recomputed at each pipeline run: silver
# lines + header, SK-resolved against the gold dims, trimmed to the
# KPI-required columns. A batch temporary view - the engine reads it at
# each run (ingestion-time snapshot versioning). No driver-side actions.
@dp.temporary_view(name="fact_order_items_snapshot")
def fact_order_items_snapshot():
    # 1) Line grain + denormalized order-header fields.
    lines = (
        spark.read.table(SILVER_ORDER_ITEMS)
        .alias("oi")
        .join(
            spark.read.table(SILVER_ORDERS).alias("o"),
            F.col("oi.order_id") == F.col("o.order_id"),
            "left",
        )
        .select(
            F.col("oi.order_item_id").alias("order_item_id"),
            F.col("oi.order_id").alias("order_id"),
            F.col("oi.product_id").alias("product_id"),
            F.col("oi.supplier_id").alias("supplier_id"),
            F.col("oi.quantity").alias("quantity"),
            F.col("oi.unit_price").alias("unit_price"),
            F.col("oi.line_total").alias("line_total"),
            F.col("o.buyer_company_id").alias("buyer_company_id"),
            F.col("o.customer_id").alias("customer_id"),
            F.col("o.order_date").alias("order_date"),
            F.col("o.order_date_valid").alias("order_date_valid"),
            F.col("o.status").alias("status"),
            F.col("o.channel").alias("channel"),
            F.col("o.created_at").alias("order_created_at"),
        )
    )

    # 2) SCD2 as-of resolution (company, product): the version open at the
    #    order's created_at, falling back to the dim's FIRST version when
    #    the order predates all versions. Exactly one version matches
    #    per line (pre-flight-verified); max() is a safe one-row fold.
    def resolve_asof_sk(dim, fk_column, business_key, surrogate_key):
        dim = dim.withColumn(
            "_first_start",
            F.min("__START_AT").over(Window.partitionBy(business_key)),
        )
        return (
            lines.select(
                "order_item_id",
                F.col(fk_column).alias("_bk"),
                "order_created_at",
            )
            .join(
                dim,
                (F.col("_bk") == F.col(business_key))
                & (
                    (
                        (F.col("__START_AT") <= F.col("order_created_at"))
                        & (
                            (F.col("__END_AT") > F.col("order_created_at"))
                            | F.col("__END_AT").isNull()
                        )
                    )
                    | (
                        (F.col("order_created_at") < F.col("_first_start"))
                        & (F.col("__START_AT") == F.col("_first_start"))
                    )
                ),
                "left",
            )
            .groupBy("order_item_id")
            .agg(F.max(surrogate_key).alias(surrogate_key))
        )

    company_sk = resolve_asof_sk(
        spark.read.table(f"{GOLD}.dim_companies"),
        "buyer_company_id",
        "company_id",
        "company_sk",
    )
    product_sk = resolve_asof_sk(
        spark.read.table(f"{GOLD}.dim_products"),
        "product_id",
        "product_id",
        "product_sk",
    )

    # 3) Assemble: lines + resolved keys, date_sk smart key (YYYYMMDD,
    #    NULL where order_date_valid is false/NULL - retained rows),
    #    final KPI-required column order.
    return (
        lines
        .join(company_sk, "order_item_id", "left")
        .join(product_sk, "order_item_id", "left")
        .join(
            spark.read.table(f"{GOLD}.dim_customers").select("customer_id", "customer_sk"),
            "customer_id",
            "left",
        )
        .join(
            spark.read.table(f"{GOLD}.dim_suppliers").select("supplier_id", "supplier_sk"),
            "supplier_id",
            "left",
        )
        .withColumn(
            "date_sk",
            F.when(
                F.col("order_date_valid"),
                F.year("order_date") * 10000
                + F.month("order_date") * 100
                + F.dayofmonth("order_date"),
            ).cast("int"),
        )
        .select(
            "order_item_id",
            "order_id",
            "date_sk",
            "order_date",
            "order_date_valid",
            "customer_sk",
            "company_sk",
            "product_sk",
            "supplier_sk",
            "order_created_at",
            "status",
            "channel",
            "quantity",
            "unit_price",
            "line_total",
        )
    )


# Delta-backed streaming table: the AUTO CDC target. Partitioned by the
# date key (requirements: fact_* partitioned by date). No clustering
# keys: Delta forbids CLUSTER BY together with PARTITIONED BY.
dp.create_streaming_table(
    name="fact_order_items",
    comment=(
        "Kimball transaction-grain fact: one row per order_item_id joining "
        "silver order lines to their order header, surrogate-keyed to the "
        "gold dimensions. company_sk and product_sk resolve AS-OF the order "
        "header created_at against the SCD2 dims (dim_companies, "
        "dim_products), falling back to the dimension's first version when "
        "an order predates it (all current orders: the SCD2 dims opened "
        "2026-11-18, after every order). customer_sk / supplier_sk are "
        "direct Type 1 keys. date_sk = YYYYMMDD from order_date, NULL for "
        "lines whose order_date_valid is false (null / future order dates "
        "- rows retained; net-revenue views filter on order_date_valid). "
        "status carries cancelled orders too (gross basis; net views "
        "exclude them). MERGE-loaded via AUTO CDC FROM SNAPSHOT keyed on "
        "order_item_id (idempotent retries). Partitioned by date_sk "
        "(requirements: fact_* partitioned by date)."
    ),
    partition_cols=["date_sk"],
)


# The auto-merge load: diff each run's fact snapshot against the last
# committed one and apply inserts/updates/deletes in place (Type 1 -
# facts do not version history; the SCD2 dims own that).
dp.create_auto_cdc_from_snapshot_flow(
    target="fact_order_items",
    source="fact_order_items_snapshot",  # ingestion-time snapshot versioning
    keys=["order_item_id"],
    stored_as_scd_type=1,
)